# Corrigé · Mini-défi J1-01 : tracer `estimer_priorite`

> ▶️ **Notebook autonome** : le socle (`src/helpdesk_ops`) fournit la configuration, le client Langfuse avec masquage et le classifieur de J1-01. Exécutez de haut en bas, stack démarrée.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from helpdesk_ops.donnees import CATEGORIES
from helpdesk_ops.tri import creer_classifieur, file_attente, normaliser_categorie

classifieur = creer_classifieur(llm)   # prompt | llm | StrOutputParser, comme en J1-01 section 4

In [ ]:
import time

# Mots-clés écrits sans accent : on compare des textes passés par sans_accents
MOTS_URGENCE = ("bloque", "urgent", "production", "tous les utilisateurs")


def estimer_priorite(ticket: str) -> str:
    """Règle métier volontairement simple (et donc explicable) : P1 si mot d'urgence, sinon P3."""
    texte = sans_accents(ticket).lower()   # « BLOQUÉ », « bloque » et « bloqué » sont traités pareil
    return "P1" if any(mot in texte for mot in MOTS_URGENCE) else "P3"


assert estimer_priorite("Écran BLOQUÉ") == "P1"
assert estimer_priorite("je suis bloque") == "P1"
assert estimer_priorite("Demande de souris") == "P3"


@observe(name="traiter-ticket")
def traiter_ticket_v2(ticket: str) -> dict:
    # 1. Classification LLM (inchangée)
    brute = classifieur.invoke(
        {"ticket": ticket, "categories": ", ".join(CATEGORIES)},
        config={"callbacks": [langfuse_handler]},
    )

    # 2. Garde-fou de normalisation (inchangé)
    with langfuse.start_as_current_observation(name="normalisation", as_type="guardrail", input=brute) as etape:
        categorie = normaliser_categorie(brute)
        etape.update(output=categorie)

    # 3. NOUVEAU : estimation de priorité, tracée comme un outil
    with langfuse.start_as_current_observation(name="estimer_priorite", as_type="tool", input=ticket) as outil:
        priorite = estimer_priorite(ticket)
        outil.update(output=priorite, metadata={"regle": "mots-cles-v1"})

    return {"categorie": categorie, "priorite": priorite, "file": file_attente(categorie)}


tickets_defi = [
    "URGENT : le serveur de production ERP est bloqué",
    "Je voudrais un deuxième écran pour mon poste",
    "Tous les utilisateurs du 3e étage ont perdu le réseau",
]

with propagate_attributes(
    session_id="defi-j1-corrige",
    tags=["jour-1", "defi"],
    version="classifieur-v2",       # comparable à v1 dans l'UI (filtre « Version »)
):
    for t in tickets_defi:
        debut = time.perf_counter()
        print(f"{traiter_ticket_v2(t)}  ({time.perf_counter() - debut:.1f}s)  ← {t}")

langfuse.flush()

## Ce qu'il fallait observer dans l'UI

- **Sessions → `defi-j1-corrige`** : 3 traces.
- Dans chaque trace : `traiter-ticket` → *generation* `ChatOpenAI` (vLLM, en salle) ou `ChatOllama` (sans GPU), *guardrail* `normalisation`, **tool `estimer_priorite`**.
- **Étape la plus lente :** toujours la *generation* LLM (des centaines de ms à quelques s), contre des microsecondes pour `estimer_priorite`. **Morale :** pour réduire la latence d'un agent, on optimise d'abord le **nombre et la taille des appels LLM**, pas le code Python.
- Filtre **Version = classifieur-v2** : on isole les traces de la nouvelle version. C'est la base des comparaisons A/B du jour 4.

## Pour aller plus loin (discussion)

- Sans `sans_accents`, « bloque » (tapé sans accent, très fréquent) ou « BLOQUÉ » échappent à la règle : en français, normalisez toujours avant de comparer.
- Le mot « bloqué » déclenche P1… même dans « je ne suis *pas* bloqué ». Une règle par mots-clés est **explicable** mais fragile. Un LLM serait plus fin, mais moins prévisible. **Au jour 4, on mesurera les deux.**
- Priorité P1 = impact métier. Au jour 2, on exigera une **validation humaine** (HITL) avant de créer un ticket P1.
- Cette règle est désormais rangée dans le socle : `helpdesk_ops.tri.estimer_priorite`.